In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 


current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)



In [2]:
import json
import openai
import numpy as np
import pandas as pd
from tqdm import tqdm
from time import time
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
openai.api_key = os.environ["OPENAI_API_KEY"]

BASEPATH = os.environ['BASEPATH']

In [3]:
def merge_sections(abstract):
    relevant_sections = ['Introduction', 'Methods', 'Results', 'Conclusion']
    merged_sections = ''
    for section in abstract['sections']:
        if section['label'] in relevant_sections:
            merged_sections = '\n'.join([merged_sections, section['markdown']])
    return merged_sections

def category_extraction(dimension, analysis_dict, dimension_categories, categories_chain, dimension_short=None):
    if dimension_short is None:
        dimension_short = dimension.split(' - ')[1]

    elif dimension_short == 'temporal':
        dimension_short = 'Temporal Scale'
    
    elif dimension_short == 'spatial':
        dimension_short = 'Spatial Scale'

    chain_input = {"dimension": dimension_short, "categories": dimension_categories[dimension_short], "analysis": analysis_dict[dimension]}
    extracted_dict = safe_dictionary_extraction(dimension_categories[dimension_short], chain_input, categories_chain, 3, 0.2)
    return extracted_dict


In [4]:
submissions_file = '/media/mario/SSD/Projects/ohbm2026/ui/data/abstracts.detail.json'

with open(submissions_file, 'r') as f:
    submissions = json.load(f)['abstracts']
submission_ids = list(submissions.keys())

abstracts = [merge_sections(submissions[submission_id]) for submission_id in submission_ids]

In [5]:
llm = ChatOpenAI(temperature=1.0, model_name='gpt-5.2')

DIMENSIONS_PROMPT = PromptTemplate(
    input_variables=["abstract"],
    template="""
            You are an expert in neuroscience and scientific text analysis. 
            You are provided a neuroscientific abstract.
            
            Your task is to identify **key neuroscience dimensions** that best describe the abstract, guided by the following 9 dimensions:

            1. Appliedness: The extent to which the research is basic science (fundamental mechanisms) or applied (translational, clinical, legal, neuroeconomics, method development, advancing technology).
            2. Modality: The sensory and/or motor modality under investigation (e.g., visual, auditory, gustatory, somatosensory, motor, sensorimotor, multimodal).
            3. Spatiotemporal Scale: The spatial (e.g., molecular, cellular, circuit, region, systems, whole-brain) and temporal (e.g., microsecond, millisecond, second, minute, hour, day, week, month, year, lifetime) scale of the research.
            4. Cognitive Complexity: The level of cognitive complexity under investigation from low level (e.g., sensory processing, motor control) to high level (e.g., language, decision making, social cognition).
            5. Species: The species under investigation (e.g., human, non-human primate, rodent, drosophila, zebrafish, C. elegans).
            6. Theory Engagement: The extent to which the research is theory-driven (hypothesis testing) or data-driven (exploratory, descriptive).
            7. Theory Scope: The scope of the theory under investigation, ranging from specific mechanisms to broad overarching theories of the brain. Intermediate between these are theories focusing on pathophysiology of a specific disorder and highly influential theories with narrow domain coverage. Indiciate the specific unifying theoretical frameworks (e.g., Predictive Coding, Critical Brain Hypothesis, Communication through Coherence, Free Energy Principle, Active Inference, Global Neuronal Workspace, Integrated Information Theory, etc.) if applicable (else, no general theory). There might be more than one framework.
            8. Methodological Approach: The methodological approach used in the research (e.g., experimental, computational, theoretical, modeling, simulation, data analysis, review, meta-analysis, etc.). Identify specific methods if applicable (e.g., optogenetics, fMRI, EEG, MEG, TMS, lesion studies, single-unit recordings, etc.). There might be more than one method.
            9. Interdisciplinarity: The extent to which the research is interdisciplinary, combining methods and concepts from multiple fields (e.g., medicine, biology, chemsitry, psychology, computer science, physics, engineering, mathematics, philosophy).

            Examples are not exhaustive and an abstract may contain multiple dimensions.

            **Output Format:**

            Please present your findings in **JSON format** with the following structure:
            {{
                "Dimension 1 - Appliedness": "Brief assessment of the research's appliedness.",
                "Dimension 2 - Modality": "Brief overview of the sensory and/or motor modality under investigation.",
                "Dimension 3 - Spatiotemporal Scale": "Brief description of the spatial and temporal scale of the research.",
                "Dimension 4 - Cognitive Complexity": "Brief assessment of the research's cognitive complexity.",
                "Dimension 5 - Species": "Brief overview of the species under investigation.",
                "Dimension 6 - Theory Engagement": "Brief assessment of the research's theory engagement.",
                "Dimension 7 - Theory Scope": "Brief assessment of the research's theory scope. Identify specific theoretical frameworks if applicable. There might be more than one framework. Not all articles need to fall under any specific framework, but a significant portion should (one or two are not enough).",
                "Dimension 8 - Methodological Approach": "Brief overview of the methodological approach used in the research. Identify specific methods if applicable. There might be more than one method.",
                "Dimension 9 - Interdisciplinarity": "Brief assessment of the research's interdisciplinarity."
            }}

            ```

            **Instructions:**
            - **Accuracy is crucial**: Ensure all information is directly supported by the provided abstract. Do not include information not present in the abstract or make external assumptions.

            - **Clarity and Precision**: Assessments and descriptions should be clear and accurately reflect the content of the abstract.

            - **Conciseness**: Do not include any additional text or explanations beyond the specified JSON output. Do not generate more output than necessary.

            - **Compliance**: Return the JSON file even when you did not receive any abstracts. Just say not applicable for all dimensions.

            **Here is the abstract:**
            {abstract}""",
)

CATEGORIES_PROMPT = PromptTemplate(
    input_variables=["dimension", "categories", "analysis"],
    template="""
            You are an expert in neuroscience. 
            You are provided with an analysis of research within a neuroscientific abstract along the following 9 dimensions:

            1. Appliedness: The extent to which the research is basic science (fundamental) or applied in one of several ways.
            2. Modality: The sensory and/or motor modality under investigation.
            3. Spatiotemporal Scale: The spatial and temporal scale of the research. Can vary from microscale to macroscale for both space and time.
            4. Cognitive Complexity: The level of cognitive complexity under investigation from low level (e.g., sensory processing, motor control) to high level (e.g., language, decision making, social cognition).
            5. Species: The species under investigation.
            6. Theory Engagement: The extent to which the research is theory-driven (hypothesis testing) or data-driven (exploratory, descriptive).
            7. Theory Scope: The scope of the theory under investigation. Overarching Framework: In neuroscience, an overarching framework is a broad theoretical approach that aims to explain fundamental principles of brain function across multiple cognitive and neural domains.
                            Domain Framework: In neuroscience, a domain framework is an integrative theory focusing on a specific subfield, offering cohesive principles for that area.
                            Disease-specific Framework: In neuroscience, a disease-specific framework is a theory that details the neural causes, mechanisms, and manifestations of a particular neurological or psychiatric condition.
                            Micro Theory: In neuroscience, a micro theory is a narrowly scoped, mechanistic account or model that explains one specific process or phenomenon in the brain.
            8. Methodological Approach: The methodological approach used in the research. Experimental: Studies in which researchers deliberately manipulate one or more variables under controlled conditions to test for causal effects.
                                        Observational (Correlational / Descriptive): Studies that measure variables in naturally occurring settings without introducing any active intervention, focusing on describing or correlating observed phenomena.
                                        Computational / coding: Studies that construct or test mathematical, algorithmic, or simulation-based models to predict, explain, or interpret empirical data or biological processes.
                                        Theoretical / Conceptual: Work that develops, refines, or critiques conceptual frameworks and theories without generating new empirical data or running computational simulations.
                                        Meta-Analytic / Systematic Review: Research that synthesizes and reanalyzes existing primary studies, systematically aggregating findings using quantitative (meta-analysis) or rigorous protocol-based (systematic review) methods.
            9. Interdisciplinarity: The extent to which the research is interdisciplinary, combining methods and concepts from multiple fields. From low (confined to a single discipline) to very high (incorporating multiple disciplines in a transcdisciplinary manner).
                                    Multidisciplinary: Multiple disciplines study the same problem in parallel, each applying its own methods and perspectives but with little cross-integration.
                                    Interdisciplinary: Researchers from different disciplines integrate theories, methods, or data to create shared frameworks or solutions that transcend any single field.
                                    Transdisciplinary: Collaboration goes beyond standard academic boundaries, involving non-academic stakeholders or merging disciplines so completely that new fields or holistic approaches emerge.

            Your task is to focus solely on the dimension of {dimension} and provide a binary indication ("yes" or "no") of whether the research within the abstract falls within the specified categories.
            Here are the categories for this dimension:
            {categories}

            **Output Format:**

            Please present your findings in **JSON format** with the following structure:
            {{
                "Category 1": "yes" / "no",
                "Category 2": "yes" / "no",
                "Category 3": "yes" / "no",
                ...
                category n: "yes" / "no"
            }}

            ```

            **Instructions:**
            - **Accuracy is crucial**: Ensure all information is directly supported by the provided analysis. Do not include information not present in the analysis or make external assumptions.

            - **Consisteny**: Ensure that the evaluation of the dimension does not contradict the provided analysis (including other dimensions).
            
            - **Focus and Precision**: Only evaluate the dimension of {dimension} and provide a binary response for each category. Do not include any additional information or explanations.

            - **Proper Category Naming**: Ensure that the categories are named correctly and accurately reflect the content of the analysis. ONLY use the provided categories and replace Category 1, Category 2, etc. with the actual category names.

            - **Binary Response**: Ensure that the response for each category is binary (yes or no) and does not include any other text or explanations.


            **Here is the analysis of the abstract along the dimension of {dimension}:**
            {analysis}""",
)

dimensions_chain = DIMENSIONS_PROMPT | llm | SimpleJsonOutputParser()
categories_chain = CATEGORIES_PROMPT | llm | SimpleJsonOutputParser()

In [6]:
# Dimensions (qualitative)

required_fields = [
                'Dimension 1 - Appliedness', 'Dimension 2 - Modality',
                'Dimension 3 - Spatiotemporal Scale',
                'Dimension 4 - Cognitive Complexity', 'Dimension 5 - Species',
                'Dimension 6 - Theory Engagement',
                'Dimension 7 - Theory Scope',
                'Dimension 8 - Methodological Approach',
                'Dimension 9 - Interdisciplinarity'
        ]

if os.path.exists(os.path.join(BASEPATH, 'OHBM2026', 'abstract_dimensions.csv')):
    abstract_dimensions_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'abstract_dimensions.csv'))
    abstract_dimensions = abstract_dimensions_df.to_dict(orient='records')
    print(f"Loaded {len(abstract_dimensions)} abstract dimensions from CSV file.")
    processed_ids = np.array([abstract_dimension['id'] for abstract_dimension in abstract_dimensions])
else:
        abstract_dimensions = []

for abstract in tqdm(abstracts, total=len(abstracts)):

        submission_id = submission_ids[abstracts.index(abstract)]
        if int(submission_id) in processed_ids:
                continue


        chain_input = {"abstract": abstract}
        extracted_dict = safe_dictionary_extraction(
                required_fields, chain_input, dimensions_chain,
                3, 0.2)

        abstract_dimensions_dict = {
                'id': submission_id,
                'Dimensions':
                '\n'.join(
                        [f'{key}: {value}' for key, value in extracted_dict.items()])
                }

        abstract_dimensions.append(abstract_dimensions_dict)

Loaded 4662 abstract dimensions from CSV file.


100%|██████████| 3333/3333 [00:00<00:00, 13402.35it/s]


In [7]:
abstract_dimensions_df = pd.DataFrame(abstract_dimensions)
abstract_dimensions_df.to_csv('/media/mario/HDD/Data/NeuroScape/OHBM2026/abstract_dimensions.csv', index=False)

In [8]:
dimension_categories = {
        'Appliedness': [
            'Fundamental', 'Translational', 'Clinical', 'Legal', 'Economic',
            'Method Development', 'Technological Exploitation'
        ],
        'Modality': [
            'Auditory', 'Visual', 'Olfactory', 'Gustatory', 'Somatosensory',
            'Multimodal', 'Visuomotor', 'Sensorimotor', 'Motor', 'Emotional',
            'Behavioral', 'Cognitive'
        ],
        'Spatial Scale': [
            'Molecular', 'Cellular', 'Circuit', 'Region', 'Systems',
            'Whole-brain'
        ],
        'Temporal Scale': [
            'Microsecond', 'Millisecond', 'Second', 'Minute', 'Hour', 'Day',
            'Week', 'Month', 'Year', 'Lifetime'
        ],
        'Cognitive Complexity':
        ['Low-level Sensory', 'Low-level Motor', 'Mid-level', 'High-level'],
        'Species': [
            'Human', 'Non-human primates', 'Rodents', 'Mammals', 'Birds',
            'Fish', 'Amphibians', 'Invertebrates', 'Cell cultures', 'Other'
        ],
        'Theory Engagement': ['Data-driven', 'Hypothesis-driven'],
        'Theory Scope': [
            'Overarching Framework', 'Domain Framework',
            'Disease-specific Framework', 'Micro Theory'
        ],
        'Methodological Approach': [
            'Experimental', 'Observational', 'Computational', 'Theoretical',
            'Meta-analytic'
        ],
        'Interdisciplinarity': ['Low', 'Medium', 'High', 'Very High']
    }

In [9]:
# Quantitative (categories)

# check if /media/mario/HDD/Data/NeuroScape/OHBM2026/appliedness.csv exists, if so, load it and skip the extraction process
if os.path.exists(os.path.join(BASEPATH, 'OHBM2026', 'appliedness.csv')):
    appliedness_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'appliedness.csv'))
    modality_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'modality.csv'))
    spatial_scale_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'spatial_scale.csv'))
    temporal_scale_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'temporal_scale.csv'))
    cognitive_complexity_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'cognitive_complexity.csv'))
    species_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'species.csv'))
    theory_engagement_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'theory_engagement.csv'))
    theory_scope_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'theory_scope.csv'))
    methodological_approach_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'methodological_approach.csv'))
    interdisciplinarity_df = pd.read_csv(os.path.join(BASEPATH, 'OHBM2026', 'interdisciplinarity.csv'))

    appliedness_list = appliedness_df.to_dict(orient='records')
    modality_list = modality_df.to_dict(orient='records')
    spatial_scale_list = spatial_scale_df.to_dict(orient='records')
    temporal_scale_list = temporal_scale_df.to_dict(orient='records')
    cognitive_complexity_list = cognitive_complexity_df.to_dict(orient='records')
    species_list = species_df.to_dict(orient='records')
    theory_engagement_list = theory_engagement_df.to_dict(orient='records')
    theory_scope_list = theory_scope_df.to_dict(orient='records')
    methodological_approach_list = methodological_approach_df.to_dict(orient='records')
    interdisciplinarity_list = interdisciplinarity_df.to_dict(orient='records')
    

    processed_ids = np.array([appliedness['id'] for appliedness in appliedness_list])

else:
    appliedness_list = []
    modality_list = []
    spatial_scale_list = []
    temporal_scale_list = []
    cognitive_complexity_list = []
    species_list = []
    theory_engagement_list = []
    theory_scope_list = []
    methodological_approach_list = []
    interdisciplinarity_list = []

    processed_ids = np.array([])

for index, row in tqdm(abstract_dimensions_df.iterrows(), total=abstract_dimensions_df.shape[0]):

    submission_id = row['id']

    if type(submission_id) == str:
        print(f"Submission ID {submission_id} is of type str.")

    if int(submission_id) in processed_ids:
        continue

    analysis = row['Dimensions']
    analysis_dict = {item.split(':')[0].strip(): item.split(':')[1].strip() for item in analysis.split('\n') if ': ' in item}

    appliedness = category_extraction('Dimension 1 - Appliedness', analysis_dict, dimension_categories, categories_chain)
    appliedness['id'] = row['id']

    modality = category_extraction('Dimension 2 - Modality', analysis_dict, dimension_categories, categories_chain)
    modality['id'] = row['id']

    spatial_scale = category_extraction('Dimension 3 - Spatiotemporal Scale', analysis_dict, dimension_categories, categories_chain, 'spatial')
    spatial_scale['id'] = row['id']

    temporal_scale = category_extraction('Dimension 3 - Spatiotemporal Scale', analysis_dict, dimension_categories, categories_chain, 'temporal')
    temporal_scale['id'] = row['id']

    cognitive_complexity = category_extraction('Dimension 4 - Cognitive Complexity', analysis_dict, dimension_categories, categories_chain)
    cognitive_complexity['id'] = row['id']

    species = category_extraction('Dimension 5 - Species', analysis_dict, dimension_categories, categories_chain)
    species['id'] = row['id']

    theory_engagement = category_extraction('Dimension 6 - Theory Engagement', analysis_dict, dimension_categories, categories_chain)
    theory_engagement['id'] = row['id']

    theory_scope = category_extraction('Dimension 7 - Theory Scope', analysis_dict, dimension_categories, categories_chain)
    theory_scope['id'] = row['id']

    methodological_approach = category_extraction('Dimension 8 - Methodological Approach', analysis_dict, dimension_categories, categories_chain)
    methodological_approach['id'] = row['id']

    interdisciplinarity = category_extraction('Dimension 9 - Interdisciplinarity', analysis_dict, dimension_categories, categories_chain)
    interdisciplinarity['id'] = row['id']

    appliedness_list.append(appliedness)
    modality_list.append(modality)
    spatial_scale_list.append(spatial_scale)
    temporal_scale_list.append(temporal_scale)
    cognitive_complexity_list.append(cognitive_complexity)
    species_list.append(species)
    theory_engagement_list.append(theory_engagement)
    theory_scope_list.append(theory_scope)
    methodological_approach_list.append(methodological_approach)
    interdisciplinarity_list.append(interdisciplinarity)



100%|██████████| 4662/4662 [00:00<00:00, 16904.30it/s]


In [10]:
appliedness_df = pd.DataFrame(appliedness_list)
modality_df = pd.DataFrame(modality_list)
spatial_scale_df = pd.DataFrame(spatial_scale_list)
temporal_scale_df = pd.DataFrame(temporal_scale_list)
cognitive_complexity_df = pd.DataFrame(cognitive_complexity_list)
species_df = pd.DataFrame(species_list)
theory_engagement_df = pd.DataFrame(theory_engagement_list)
theory_scope_df = pd.DataFrame(theory_scope_list)
methodological_approach_df = pd.DataFrame(methodological_approach_list)
interdisciplinarity_df = pd.DataFrame(interdisciplinarity_list)

appliedness_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'appliedness.csv'), index=False)
modality_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'modality.csv'), index=False)
spatial_scale_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'spatial_scale.csv'), index=False)
temporal_scale_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'temporal_scale.csv'), index=False)
cognitive_complexity_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'cognitive_complexity.csv'), index=False)
species_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'species.csv'), index=False)
theory_engagement_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'theory_engagement.csv'), index=False)
theory_scope_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'theory_scope.csv'), index=False)
methodological_approach_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'methodological_approach.csv'), index=False)
interdisciplinarity_df.to_csv(os.path.join(BASEPATH, 'OHBM2026', 'interdisciplinarity.csv'), index=False)

In [12]:
def get_categories(dimension_df):
    # drop the id column
    dimension_df = dimension_df.drop(columns=['id'])
    return [category for category in dimension_df.columns if dimension_df[category].values[0] == 'yes']
    

for i, submission_id in tqdm(enumerate(submission_ids), total=len(submission_ids)):

    try:
        focus_categories = get_categories(appliedness_df[appliedness_df['id'] == int(submission_id)])
        research_modality_categories = get_categories(methodological_approach_df[methodological_approach_df['id'] == int(submission_id)])
        theory_scope_categories = get_categories(theory_scope_df[theory_scope_df['id'] == int(submission_id)])
        theory_engagement_categories = get_categories(theory_engagement_df[theory_engagement_df['id'] == int(submission_id)])


        submissions[submission_id]['focus'] = focus_categories
        submissions[submission_id]['research_modality'] = research_modality_categories
        submissions[submission_id]['theory_scope'] = theory_scope_categories
        submissions[submission_id]['epistemic_basis'] = theory_engagement_categories
    except Exception as e:
        print(f"Error processing submission ID {submission_id}: {e}")

  0%|          | 0/3333 [00:00<?, ?it/s]

 99%|█████████▉| 3293/3333 [00:08<00:00, 370.82it/s]

Error processing submission ID 1248744: index 0 is out of bounds for axis 0 with size 0


100%|██████████| 3333/3333 [00:09<00:00, 366.72it/s]


In [13]:
new_submissions_file = '/media/mario/SSD/Projects/ohbm2026/ui/data/new/abstracts.detail.json'

os.makedirs(os.path.dirname(new_submissions_file), exist_ok=True)
with open(new_submissions_file, 'w') as f:
    json.dump({'abstracts': submissions}, f, indent=4)